In [1]:
!pip install pyspark -q


In [2]:
# ================================================
# Initialize Spark Session
# ================================================
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week6_Spark_Architecture_and_Optimization") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

print("Spark Session created successfully!")
print(f"Spark Version: {spark.version}")

Spark Session created successfully!
Spark Version: 4.0.3


In [4]:
# ================================================
# Step 3: Read CSV with proper schema handling
# ================================================
from pyspark.sql.functions import col, regexp_extract
from pyspark.sql.types import DoubleType

def load_csv(filepath: str):
    """
    Reads a CSV file into a Spark DataFrame.
    - header=True: First row treated as column names
    - inferSchema=True: Automatically detects data types
    - multiLine + escape: Handles embedded commas in quoted fields

    Args:
        filepath (str): Path to CSV file
    Returns:
        DataFrame: Loaded Spark DataFrame
    """
    df = spark.read.csv(
        filepath,
        header=True,
        inferSchema=True,
        multiLine=True,
        escape='"'
    )

    # Safely cast numeric columns to avoid malformed value errors
    numeric_cols = ["Sales", "Quantity", "Discount", "Profit"]
    for c in numeric_cols:
        df = df.withColumn(
            c,
            regexp_extract(col(c).cast("string"), r'^-?\d*\.?\d+', 0).cast(DoubleType())
        )

    print(f"CSV loaded successfully! Rows: {df.count()}, Columns: {len(df.columns)}")
    return df

df = load_csv("Sample - Superstore.csv")
df.show(5)
df.printSchema()

CSV loaded successfully! Rows: 9994, Columns: 21
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-100

# Celebal Technologies Summer Internship 2026
## Week 6: Spark Architecture and Optimized Data Processing
---

## Q1: Roles of Driver, Cluster Manager, and Executor

**Driver:**
- The brain of a Spark application
- Runs the `main()` function and creates the SparkContext/SparkSession
- Converts user code into a DAG of tasks
- Schedules and coordinates task execution across the cluster
- Collects results from Executors

**Cluster Manager:**
- Manages and allocates cluster resources (CPU, memory)
- Examples: YARN, Mesos, Kubernetes, or Spark Standalone
- Acts as a middleman between the Driver and Executors
- Driver requests resources from Cluster Manager; Cluster Manager assigns Executors

**Executor:**
- Worker processes launched on cluster nodes
- Execute the actual tasks assigned by the Driver
- Store data in memory/disk for caching (RDD/DataFrame persistence)
- Send results back to the Driver
- Each application has its own set of Executors

**Flow:**
Driver → requests resources from → Cluster Manager → launches → Executors → execute tasks → return results to → Driver

## Q2: Lazy Evaluation and Performance

**What is Lazy Evaluation?**
Spark does NOT execute transformations immediately when they are called. Instead, it builds a logical plan (DAG) and only executes when an **action** (like `.show()`, `.count()`, `.write()`) is triggered.

**How it improves performance:**
1. **Query optimization:** Spark's Catalyst Optimizer analyzes the full DAG before execution and applies optimizations like predicate pushdown, column pruning, and join reordering
2. **Avoids unnecessary computation:** If only 5 rows are needed (`.show(5)`), Spark won't process all 9994 rows unnecessarily
3. **Pipelining:** Multiple transformations are combined into a single stage, reducing intermediate data materialization
4. **Fault recovery:** The DAG lineage allows Spark to recompute only the lost partitions if a node fails

**Example:**
```python
df.filter(...).select(...).groupBy(...).show(5)
# Nothing executes until .show(5) is called!
# Spark optimizes the entire chain first, then executes efficiently
```

In [6]:
# ================================================
# Q3: Read CSV with header and inferSchema
# ================================================
# Already demonstrated in load_csv() function above
# This is the standard Spark command:

df_q3 = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,        # First row as header
    inferSchema=True    # Auto-detect data types
)
print("=== Q3: CSV loaded with header + inferSchema ===")
print(f"Rows: {df_q3.count()}, Columns: {len(df_q3.columns)}")

=== Q3: CSV loaded with header + inferSchema ===
Rows: 9994, Columns: 21


## Q4: CSV vs Parquet - Storage Format Comparison

| Feature | CSV | Parquet |
|---------|-----|---------|
| Storage type | Row-based | Columnar |
| File size | Large | Small (compressed) |
| Read speed | Slow (reads all columns) | Fast (reads only needed columns) |
| Schema | No schema stored | Schema embedded |
| Compression | None by default | Snappy/GZIP built-in |
| Best for | Small data, human-readable | Large-scale analytics |

**Why it matters for performance:**

**Row-based (CSV):**
- To read 2 columns from 100 columns, CSV must scan ALL columns for every row
- High I/O overhead for analytical queries

**Columnar (Parquet):**
- Only the required columns are read from disk — rest are skipped entirely
- Combined with **Predicate Pushdown**, only matching rows are loaded
- Result: dramatically less data read from disk → faster queries + less memory usage

**Example:**
Querying Sales from 9994 rows:
- CSV: reads all 21 columns × 9994 rows
- Parquet: reads only Sales column → ~95% less data read!

In [8]:
# ================================================
# Q5: Select product_id and price where category = 'Electronics'
# ================================================
# Mapped to: Product ID, Sales where Category = 'Technology'

def filter_and_select(df, category: str):
    """
    Selects product_id and price columns filtered by category.

    Args:
        df (DataFrame): Input Spark DataFrame
        category (str): Category to filter on
    Returns:
        DataFrame: Filtered and selected DataFrame
    """
    return df.filter(col("Category") == category) \
             .select("Product ID", "Sales", "Category")

print("\n=== Q5: Product ID and Sales where Category = Technology ===")
q5_result = filter_and_select(df, "Technology")
q5_result.show(5)
print(f"Total Technology rows: {q5_result.count()}")


=== Q5: Product ID and Sales where Category = Technology ===
+---------------+--------+----------+
|     Product ID|   Sales|  Category|
+---------------+--------+----------+
|TEC-PH-10002275| 907.152|Technology|
|TEC-PH-10002033| 911.424|Technology|
|TEC-PH-10001949|  213.48|Technology|
|TEC-AC-10003027|   90.57|Technology|
|TEC-PH-10004977|1097.544|Technology|
+---------------+--------+----------+
only showing top 5 rows
Total Technology rows: 1847


In [9]:
# ================================================
# Q6: Rename column and cast data type
# ================================================

def rename_and_cast(df, old_name: str, new_name: str, col_to_cast: str, new_type: str):
    """
    Renames a column and casts another column to a new data type.

    Args:
        df (DataFrame): Input Spark DataFrame
        old_name (str): Current column name
        new_name (str): New column name
        col_to_cast (str): Column to cast
        new_type (str): Target data type
    Returns:
        DataFrame: Modified DataFrame
    """
    return df.withColumnRenamed(old_name, new_name) \
             .withColumn(col_to_cast, col(col_to_cast).cast(new_type))


print("=== Q6: Rename 'Product ID' to 'product_id' and cast 'Sales' to String ===")
df_q6 = rename_and_cast(df, "Product ID", "product_id", "Sales", "string")
df_q6.select("product_id", "Sales").printSchema()
df_q6.select("product_id", "Sales").show(5)

=== Q6: Rename 'Product ID' to 'product_id' and cast 'Sales' to String ===
root
 |-- product_id: string (nullable = true)
 |-- Sales: string (nullable = true)

+---------------+--------+
|     product_id|   Sales|
+---------------+--------+
|FUR-BO-10001798|  261.96|
|FUR-CH-10000454|  731.94|
|OFF-LA-10000240|   14.62|
|FUR-TA-10000577|957.5775|
|OFF-ST-10000760|  22.368|
+---------------+--------+
only showing top 5 rows


## Q7: Lineage Graph (DAG) and Fault Tolerance

**What is DAG (Directed Acyclic Graph)?**
Spark tracks every transformation applied to a DataFrame as a DAG — a sequence of steps showing how data was derived from the source.

**How it provides fault tolerance:**
1. When a worker node fails, the data it held in memory is lost
2. Instead of restarting the entire job, Spark uses the **lineage graph** to identify which partition was lost
3. Spark **recomputes only the lost partition** by replaying the transformations from the last stable point (source file or checkpoint)
4. No data replication needed — the lineage itself acts as the recovery mechanism

**Example:**
CSV File → filter(Region='West') → groupBy(Category) → [LOST PARTITION]
Spark recomputes the lost partition by re-reading only the relevant rows from the CSV and reapplying the transformations — without restarting the full job.

**Advantage over MapReduce:**
MapReduce achieves fault tolerance by writing intermediate results to HDFS after every step. Spark avoids this disk overhead by using lineage — making recovery faster and more efficient.

In [10]:
# ================================================
# Q8: Filter status='Completed' AND amount > 1000
# ================================================
# Mapped to: Ship Mode='Second Class' AND Sales > 1000

def filter_status_and_amount(df, status: str, amount: float):
    """
    Filters DataFrame by status AND amount threshold.

    Args:
        df (DataFrame): Input Spark DataFrame
        status (str): Ship Mode value to filter on
        amount (float): Minimum Sales amount
    Returns:
        DataFrame: Filtered DataFrame
    """
    return df.filter(
        (col("Ship Mode") == status) &
        (col("Sales") > amount)
    )

print("=== Q8: Ship Mode='Second Class' AND Sales > 1000 ===")
q8_result = filter_status_and_amount(df, "Second Class", 1000)
q8_result.select("Order ID", "Ship Mode", "Sales").show(5)
print(f"Total matching rows: {q8_result.count()}")

=== Q8: Ship Mode='Second Class' AND Sales > 1000 ===
+--------------+------------+--------+
|      Order ID|   Ship Mode|   Sales|
+--------------+------------+--------+
|CA-2014-131926|Second Class| 2001.86|
|CA-2014-131926|Second Class| 1503.25|
|US-2014-106992|Second Class|3059.982|
|US-2014-106992|Second Class|2519.958|
|US-2015-161991|Second Class|  1114.4|
+--------------+------------+--------+
only showing top 5 rows
Total matching rows: 94


## Q9: Predicate Pushdown in Parquet

**What is Predicate Pushdown?**
Predicate Pushdown is an optimization technique where filter conditions (predicates) are "pushed down" to the data source level — meaning Spark applies filters **before** loading data into memory, rather than after.

**How it works with Parquet:**
1. Parquet stores **column statistics** (min, max, null count) for each row group in its footer metadata
2. When Spark executes a filter like `WHERE Sales > 1000`, it first checks the Parquet metadata
3. Row groups where `max(Sales) < 1000` are **skipped entirely** — never read from disk
4. Only relevant row groups are loaded into memory

**Impact on performance:**
- Drastically reduces I/O — only a fraction of data is read from disk
- Less memory usage — irrelevant data never enters Spark's memory
- Faster query execution — fewer partitions to process

**Example:**
```python
# Without Predicate Pushdown (CSV):
# Reads ALL 9994 rows → loads into memory → then filters
df_csv.filter(col("Sales") > 1000).show()

# With Predicate Pushdown (Parquet):
# Checks metadata → skips row groups where max(Sales) < 1000
# → loads only relevant rows → applies filter
df_parquet.filter(col("Sales") > 1000).show()
```

**Result:** Parquet with Predicate Pushdown can be 10-100x faster than CSV for large datasets with selective filters.

In [11]:
# ================================================
# Q10: Add new column final_price = base_price * 1.18 (18% tax)
# ================================================

def add_tax_column(df, base_col: str, tax_rate: float = 1.18):
    """
    Adds a new column with tax applied to base price.

    Args:
        df (DataFrame): Input Spark DataFrame
        base_col (str): Column representing base price
        tax_rate (float): Tax multiplier (default 1.18 = 18% tax)
    Returns:
        DataFrame: DataFrame with new final_price column
    """
    return df.withColumn("final_price", col(base_col) * tax_rate)

print("\n=== Q10: Add final_price = Sales * 1.18 (18% tax) ===")
df_q10 = add_tax_column(df, "Sales")
df_q10.select("Sales", "final_price").show(5)


=== Q10: Add final_price = Sales * 1.18 (18% tax) ===
+--------+------------------+
|   Sales|       final_price|
+--------+------------------+
|  261.96|309.11279999999994|
|  731.94|          863.6892|
|   14.62|           17.2516|
|957.5775|        1129.94145|
|  22.368|26.394239999999996|
+--------+------------------+
only showing top 5 rows


## Q11: Transformations vs Actions

**Transformations:**
- Operations that create a new DataFrame from an existing one
- **Lazy** — not executed immediately, just added to DAG
- Examples:
  - `.filter()` — filters rows based on condition
  - `.select()` — selects specific columns
  - `.withColumn()` — adds/modifies a column
  - `.groupBy()` — groups data by a column

**Actions:**
- Operations that trigger actual execution of the DAG
- Return results to the Driver or write to storage
- Examples:
  - `.show()` — displays rows on screen
  - `.count()` — returns total row count
  - `.collect()` — returns all rows to Driver as Python list
  - `.write()` — saves DataFrame to storage

**Key Difference:**
| | Transformations | Actions |
|--|----------------|---------|
| Execution | Lazy (deferred) | Immediate |
| Returns | New DataFrame | Result/Output |
| DAG | Adds to DAG | Triggers DAG execution |
| Examples | filter, select, groupBy | show, count, write |

In [12]:
# ================================================
# Q12: Load Parquet, filter null user_id, save as CSV
# ================================================

def save_as_parquet(df, path: str):
    """
    Saves DataFrame as Parquet file.

    Args:
        df (DataFrame): Input Spark DataFrame
        path (str): Output path
    """
    df.write.mode("overwrite").parquet(path)
    print(f"Parquet saved at: {path}")


def load_parquet_filter_save_csv(parquet_path: str, output_path: str, filter_col: str):
    """
    Loads Parquet file, filters null values, saves as CSV.

    Args:
        parquet_path (str): Input Parquet path
        output_path (str): Output CSV path
        filter_col (str): Column to check for nulls
    """
    df_parquet = spark.read.parquet(parquet_path)
    df_filtered = df_parquet.filter(col(filter_col).isNotNull())
    df_filtered.write.mode("overwrite").csv(output_path, header=True)
    print(f"Filtered CSV saved at: {output_path}")
    return df_filtered

# Step 1: Save current df as Parquet first
print("=== Saving DataFrame as Parquet ===")
save_as_parquet(df, "/content/superstore.parquet")

# Step 2: Load Parquet, filter nulls in Customer ID, save as CSV
print("\n=== Q12: Load Parquet → Filter nulls → Save as CSV ===")
df_q12 = load_parquet_filter_save_csv(
    "/content/superstore.parquet",
    "/content/output_q12",
    "Customer ID"
)
print(f"Rows after filtering nulls: {df_q12.count()}")
df_q12.show(5)

=== Saving DataFrame as Parquet ===
Parquet saved at: /content/superstore.parquet

=== Q12: Load Parquet → Filter nulls → Save as CSV ===
Filtered CSV saved at: /content/output_q12
Rows after filtering nulls: 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|  

## Q13: Client Mode vs Cluster Mode

| Feature | Client Mode | Cluster Mode |
|---------|-------------|--------------|
| Driver location | Runs on the **client machine** (your laptop/gateway) | Runs **inside the cluster** (on a worker node) |
| Network dependency | Client must stay connected throughout job | Client can disconnect after submission |
| Best for | Interactive development, debugging, notebooks | Production batch jobs |
| Latency | Higher (results travel back to client) | Lower (Driver is close to Executors) |
| Resource usage | Client machine resources used for Driver | Cluster resources used for Driver |

**Client Mode:**
- You submit a job from your local machine
- Driver runs on YOUR machine
- If your machine disconnects → job fails
- Used in: Jupyter notebooks, PySpark shell, development

**Cluster Mode:**
- Driver runs inside the cluster on a worker node
- You can close your laptop after submitting
- More resilient for long-running production jobs
- Used in: YARN, Kubernetes, production pipelines

In [14]:
# ================================================
# Q14: Filter region='North' OR priority='High'
# ================================================
# Mapped to: Region='West' OR Segment='Corporate'

def filter_region_or_priority(df, region: str, priority: str):
    """
    Filters DataFrame where region matches OR priority/segment matches.
    Demonstrates OR condition filtering in Spark.

    Args:
        df (DataFrame): Input Spark DataFrame
        region (str): Region value to filter
        priority (str): Segment/priority value to filter
    Returns:
        DataFrame: Filtered DataFrame
    """
    return df.filter(
        (col("Region") == region) | (col("Segment") == priority)
    )

print("=== Q14: Region='West' OR Segment='Corporate' ===")
q14_result = filter_region_or_priority(df, "West", "Corporate")
q14_result.select("Order ID", "Region", "Segment", "Sales").show(5)
print(f"Total matching rows: {q14_result.count()}")

=== Q14: Region='West' OR Segment='Corporate' ===
+--------------+------+---------+-------+
|      Order ID|Region|  Segment|  Sales|
+--------------+------+---------+-------+
|CA-2016-138688|  West|Corporate|  14.62|
|CA-2014-115812|  West| Consumer|  48.86|
|CA-2014-115812|  West| Consumer|   7.28|
|CA-2014-115812|  West| Consumer|907.152|
|CA-2014-115812|  West| Consumer| 18.504|
+--------------+------+---------+-------+
only showing top 5 rows
Total matching rows: 5263


## Q15: .show(5) vs .collect() on Large Datasets

**Why .show(5) is safer:**

| | .show(5) | .collect() |
|--|----------|------------|
| Data returned | Only 5 rows displayed | ALL rows returned to Driver |
| Memory usage | Minimal | Entire dataset loaded into Driver memory |
| Risk | None | OutOfMemoryError on large datasets |
| Network | Minimal data transfer | Huge data transfer to Driver |
| Use case | Exploration, debugging | Only for very small datasets |

**Problem with .collect() on large data:**
- On a multi-terabyte dataset, `.collect()` tries to bring ALL data to the Driver node
- Driver has limited memory (e.g., 16GB) — cannot hold terabytes
- Result: **OutOfMemoryError** → job crashes → cluster instability

**Best practices:**
- Use `.show(n)` for quick data preview
- Use `.count()` for row count instead of `len(df.collect())`
- Use `.write()` to save results instead of collecting to Driver
- Only use `.collect()` when dataset is confirmed to be small (after filtering)

**Example:**
```python
# WRONG on large data - crashes with OutOfMemoryError
all_data = df.collect()  

# CORRECT - safe preview
df.show(5)

# CORRECT - save large results to storage
df.write.parquet("output/results")
```

In [15]:
# ================================================
# Final Pipeline: Read → Transform → Filter → Write
# ================================================
from pyspark.sql.functions import to_timestamp

def run_complete_pipeline(input_path: str, output_csv: str, output_parquet: str):
    """
    Complete end-to-end data pipeline:
    1. Read CSV with schema handling
    2. Clean data (remove duplicates, handle nulls)
    3. Transform (add final_price, cast Order Date)
    4. Filter (Completed orders with Sales > 500)
    5. Write output as both CSV and Parquet

    Args:
        input_path (str): Source CSV path
        output_csv (str): Output CSV path
        output_parquet (str): Output Parquet path
    Returns:
        DataFrame: Final processed DataFrame
    """
    print("=" * 50)
    print("PIPELINE STARTED")
    print("=" * 50)

    # Step 1: Read
    print("\nStep 1: Reading CSV...")
    raw_df = load_csv(input_path)

    # Step 2: Clean
    print("\nStep 2: Cleaning data...")
    clean_df = raw_df.dropDuplicates()
    clean_df = clean_df.na.fill(0, subset=["Sales", "Discount"])
    print(f"Rows after cleaning: {clean_df.count()}")

    # Step 3: Transform
    print("\nStep 3: Transforming data...")
    transformed_df = clean_df \
        .withColumn("final_price", col("Sales") * 1.18) \
        .withColumn("event_time", to_timestamp(col("Order Date"), "M/d/yyyy")) \
        .withColumnRenamed("Product ID", "product_id")

    # Step 4: Filter
    print("\nStep 4: Filtering (Sales > 500 AND Region = West)...")
    filtered_df = transformed_df.filter(
        (col("Sales") > 500) & (col("Region") == "West")
    )
    print(f"Rows after filtering: {filtered_df.count()}")

    # Step 5: Write CSV
    print(f"\nStep 5a: Saving as CSV to {output_csv}...")
    filtered_df.write.mode("overwrite").csv(output_csv, header=True)
    print("CSV saved!")

    # Step 5b: Write Parquet
    print(f"\nStep 5b: Saving as Parquet to {output_parquet}...")
    filtered_df.write.mode("overwrite").parquet(output_parquet)
    print("Parquet saved!")

    print("\n" + "=" * 50)
    print("PIPELINE COMPLETED SUCCESSFULLY!")
    print("=" * 50)

    return filtered_df


# Execute pipeline
final_df = run_complete_pipeline(
    "Sample - Superstore.csv",
    "/content/output/pipeline_output_csv",
    "/content/output/pipeline_output_parquet"
)

print("\n=== Final Pipeline Output (Top 5 rows) ===")
final_df.select("Order ID", "Region", "Sales", "final_price", "event_time").show(5)

PIPELINE STARTED

Step 1: Reading CSV...
CSV loaded successfully! Rows: 9994, Columns: 21

Step 2: Cleaning data...
Rows after cleaning: 9994

Step 3: Transforming data...

Step 4: Filtering (Sales > 500 AND Region = West)...
Rows after filtering: 374

Step 5a: Saving as CSV to /content/output/pipeline_output_csv...
CSV saved!

Step 5b: Saving as Parquet to /content/output/pipeline_output_parquet...
Parquet saved!

PIPELINE COMPLETED SUCCESSFULLY!

=== Final Pipeline Output (Top 5 rows) ===
+--------------+------+--------+------------------+-------------------+
|      Order ID|Region|   Sales|       final_price|         event_time|
+--------------+------+--------+------------------+-------------------+
|CA-2016-105081|  West| 1747.25|          2061.755|2016-12-25 00:00:00|
|CA-2015-120362|  West|  912.75|1077.0449999999998|2015-09-14 00:00:00|
|US-2016-155768|  West| 1112.94|         1313.2692|2016-12-01 00:00:00|
|US-2015-126214|  West| 1618.37|1909.6765999999998|2015-12-21 00:00:00|


In [16]:
# ================================================
# CSV vs Parquet Performance Comparison
# ================================================
import time

def measure_read_time(format_type: str, path: str, filter_col: str, filter_val: float):
    """
    Measures read + filter time for CSV vs Parquet.

    Args:
        format_type (str): 'csv' or 'parquet'
        path (str): File path
        filter_col (str): Column to filter on
        filter_val (float): Filter value
    Returns:
        tuple: (row_count, elapsed_time)
    """
    start = time.time()
    if format_type == "csv":
        df_temp = spark.read.csv(path, header=True, inferSchema=True)
    else:
        df_temp = spark.read.parquet(path)
    count = df_temp.filter(col(filter_col) > filter_val).count()
    elapsed = round(time.time() - start, 3)
    return count, elapsed


print("=== CSV vs Parquet Performance Comparison ===\n")

# CSV read time
csv_count, csv_time = measure_read_time(
    "csv", "Sample - Superstore.csv", "Sales", 500
)
print(f"CSV  → Rows: {csv_count} | Time: {csv_time}s")

# Parquet read time
parquet_count, parquet_time = measure_read_time(
    "parquet", "/content/superstore.parquet", "Sales", 500
)
print(f"Parquet → Rows: {parquet_count} | Time: {parquet_time}s")

if parquet_time < csv_time:
    speedup = round(csv_time / parquet_time, 2)
    print(f"\nParquet is {speedup}x faster than CSV!")
else:
    print("\nNote: On small datasets, difference may be minimal.")
    print("On TB-scale data, Parquet is significantly faster due to columnar storage + predicate pushdown.")

=== CSV vs Parquet Performance Comparison ===

CSV  → Rows: 1150 | Time: 1.46s
Parquet → Rows: 1162 | Time: 1.844s

Note: On small datasets, difference may be minimal.
On TB-scale data, Parquet is significantly faster due to columnar storage + predicate pushdown.


---
# Summary & Key Insights
---

## Pipeline Execution Results
- Input: 9994 rows, 21 columns (Superstore CSV)
- After cleaning: 9994 rows (no duplicates found)
- After filtering (Sales > 500 AND Region = West): 374 rows
- Output saved as both CSV and Parquet formats

## Performance Comparison: CSV vs Parquet
- CSV read time: 1.46s
- Parquet read time: 1.844s
- Note: On small datasets (9994 rows), difference is minimal
- On TB-scale data, Parquet is 10-100x faster due to:
  - Columnar storage (reads only needed columns)
  - Predicate Pushdown (skips irrelevant row groups)
  - Built-in compression (smaller file size)

## Key Concepts Demonstrated
| Concept | Demonstrated |
|---------|-------------|
| Spark Architecture | Driver, Cluster Manager, Executor roles explained |
| Lazy Evaluation | Transformations build DAG, Actions trigger execution |
| CSV vs Parquet | Performance comparison + theory |
| Predicate Pushdown | Parquet metadata-based row skipping |
| Transformations | filter(), select(), withColumn(), withColumnRenamed() |
| Actions | show(), count(), write() |
| Fault Tolerance | DAG lineage graph recomputation |
| Client vs Cluster Mode | Deployment mode comparison |
| Data Pipeline | Read → Transform → Filter → Write |

## Best Practices Applied
- Used `.show()` instead of `.collect()` for large data exploration
- Used modular functions with docstrings for reusability
- Applied `regexp_extract()` for safe numeric parsing
- Saved output in both CSV (readable) and Parquet (performant) formats
- Used `mode("overwrite")` to handle re-runs **gracefully**